# 📸 Snap.py v4.1 - Web Snapshot Tool for Google Colab

**Backup stron internetowych z pełnymi zasobami, screenshotami i crawlingiem**

## Funkcje:
- ✅ **Full mode**: HTML + wszystkie zasoby (CSS, JS, obrazy, fonty) + screenshot
- ✅ **Screenshots mode**: Tylko pełne screenshoty stron (szybki tryb)
- ✅ **Crawl mode**: Automatyczne wykrywanie podstron z sitemap + linków
- ✅ **Clean modes**: Agresywne usuwanie popupów/cookies przed backupem
- 💾 Zapis bezpośrednio na **Google Drive**
- 🎯 Obsługa sliderów, lazy-load, animacji, preloaderów
- 🍪 Automatyczne akceptowanie cookies (RODO/GDPR)
- 🚀 Blokada analytics/tracking dla szybkości

---

## 🔧 Krok 1: Instalacja zależności

Uruchom tę komórkę **tylko raz** przy pierwszym użyciu (lub po resecie runtime).

In [ ]:
# Instalacja zależności
!pip install playwright requests ipywidgets
!playwright install chromium
!playwright install-deps chromium

print("✅ Instalacja zakończona!")

## 📁 Krok 2: Montowanie Google Drive

Po uruchomieniu poniższej komórki postępuj zgodnie z instrukcjami, aby autoryzować dostęp do Drive.

In [ ]:
from google.colab import drive
import os

# Montuj Google Drive
drive.mount('/content/drive')

# Ustaw ścieżkę do folderu z backupami
BACKUP_FOLDER = "/content/drive/MyDrive/Web_Backups"
os.makedirs(BACKUP_FOLDER, exist_ok=True)

print(f"📁 Backup folder: {BACKUP_FOLDER}")

## ⚙️ Krok 3: Konfiguracja

Dostosuj ustawienia według potrzeb:

In [ ]:
# KONFIGURACJA

# Liczba równoległych workerów (Colab: zalecane 1-2)
WORKERS = 1

# Rozdzielczość viewport
VIEWPORT_WIDTH = 1440
VIEWPORT_HEIGHT = 900

# Maksymalna wysokość screenshotu (px)
MAX_SCREENSHOT_HEIGHT = 15000

# Maksymalna liczba stron w trybie crawl
MAX_PAGES_CRAWL = 50

# Blokuj analytics/tracking (szybsze działanie)
BLOCK_ANALYTICS = True

# Folder wyjściowy na Google Drive
OUTPUT_DIR = BACKUP_FOLDER

print("✅ Konfiguracja zapisana")
print(f"   Workers: {WORKERS}")
print(f"   Viewport: {VIEWPORT_WIDTH}x{VIEWPORT_HEIGHT}")
print(f"   Max screenshot height: {MAX_SCREENSHOT_HEIGHT}px")
print(f"   Max pages (crawl): {MAX_PAGES_CRAWL}")
print(f"   Block analytics: {BLOCK_ANALYTICS}")
print(f"   Output: {OUTPUT_DIR}")

## 🌐 Krok 4: Lista stron do backupu

Edytuj poniższą listę - jedna strona na linię. Linie zaczynające się od `#` są ignorowane.

In [ ]:
# LISTA STRON DO BACKUPU
# Edytuj tę komórkę - dodaj lub usuń URL-e

URL_LIST = """
https://ecomess.pl/produkty/picoflux-air/
https://ecomess.pl/produkty/antena-mbwblue/
https://ecomess.pl/produkty/ecors3/
https://ecomess.pl/produkty/sensostar-e/
https://ecomess.pl/kategorie/odczyt-zdalny/
# https://ta-strona-jest-wylaczona.pl
"""

# Parse URLs
urls = [line.strip() for line in URL_LIST.strip().split('\n') 
        if line.strip() and not line.strip().startswith('#')]

print(f"📋 Znaleziono {len(urls)} stron do backupu:")
for i, url in enumerate(urls, 1):
    print(f"   {i}. {url}")

## 🚀 Krok 5: Wybór trybu i uruchomienie

Dostępne tryby:
- `'full'` - HTML + assets + screenshot (pełny backup)
- `'screenshots'` - tylko screenshoty (szybki)
- `'crawl'` - automatyczne wykrywanie podstron
- `'clean-full'` - jak full, ale z agresywnym usuwaniem popupów
- `'clean-screenshots'` - jak screenshots, ale z czyszczeniem popupów
- `'clean-crawl'` - jak crawl, ale z czyszczeniem popupów

In [ ]:
# WYBÓR TRYBU
# Odkomentuj wybraną opcję:

MODE = 'full'              # Pełny backup (HTML + assets + screenshot)
# MODE = 'screenshots'      # Tylko screenshoty
# MODE = 'crawl'            # Auto-detect podstron
# MODE = 'clean-full'       # Full + agresywne czyszczenie popupów
# MODE = 'clean-screenshots' # Screenshots + czyszczenie
# MODE = 'clean-crawl'      # Crawl + czyszczenie

print(f"🎯 Wybrany tryb: {MODE}")

## ▶️ Uruchomienie backupu

Uruchom poniższą komórkę, aby rozpocząć proces backupu.

In [ ]:
# GŁÓWNY SKRYPT SNAP.PY - ZOPTYMALIZOWANY DLA COLAB

import logging
import re
import shutil
import sys
import time
import zipfile
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Tuple
from urllib.parse import urlparse, urljoin, urldefrag

import requests
from playwright.sync_api import sync_playwright

# ─── KONFIGURACJA ─────────────────────────────────────────────────────────────
_CFG = {
    'workers': WORKERS,
    'block_analytics': BLOCK_ANALYTICS,
    'viewport_width': VIEWPORT_WIDTH,
    'viewport_height': VIEWPORT_HEIGHT,
    'max_screenshot_height': MAX_SCREENSHOT_HEIGHT,
    'max_pages': MAX_PAGES_CRAWL,
}

NORMAL_USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/121.0.0.0 Safari/537.36"
)

ASSET_EXTENSIONS = {
    '.css', '.js',
    '.png', '.jpg', '.jpeg', '.gif', '.svg', '.webp', '.avif', '.ico',
    '.woff', '.woff2', '.ttf', '.eot', '.otf',
    '.mp4', '.webm', '.ogg', '.mp3',
    '.json',
}

CONTENT_TYPE_MAP = {
    'image/jpeg': '.jpg', 'image/png': '.png', 'image/gif': '.gif',
    'image/webp': '.webp', 'image/avif': '.avif', 'image/svg+xml': '.svg',
    'image/x-icon': '.ico', 'image/vnd.microsoft.icon': '.ico',
    'font/woff': '.woff', 'font/woff2': '.woff2', 'font/ttf': '.ttf',
    'font/otf': '.otf', 'application/font-woff': '.woff',
    'application/font-woff2': '.woff2', 'application/x-font-ttf': '.ttf',
    'text/css': '.css', 'application/javascript': '.js', 'text/javascript': '.js',
    'application/json': '.json',
}

# ─── INIT SCRIPT - wstrzykiwany przed nawigacją ──────────────────────────────
_INIT_SCRIPT = r"""
(() => {
    const css = `
        *, *::before, *::after {
            animation-duration: 0.001s !important;
            animation-delay: 0s !important;
            transition-duration: 0.001s !important;
            transition-delay: 0s !important;
            animation-iteration-count: 1 !important;
        }
        .wow, [class*="animate__"], [data-aos], .elementor-invisible,
        .probootstrap-animate, .et-waypoint, .et_animated,
        .fl-animation, .u-animate, [data-animate-effect], [data-animate],
        [data-animation], [data-appear], [data-scroll-animation], [data-sal],
        [class*="entrance-anim"], [class*="scroll-anim"],
        [class*="reveal-anim"], [class*="fade-in"]:not(body):not(html),
        [class*="slide-in"]:not(body):not(html),
        [class*="zoom-in"]:not(body):not(html),
        [class*="-animate"]:not(html):not(body):not(script):not(style),
        [class*="_animate"]:not(html):not(body):not(script):not(style),
        [class*="_animated"]:not(html):not(body):not(script):not(style),
        .animated:not(html):not(body):not(script):not(style) {
            visibility: visible !important;
            opacity: 1 !important;
            transform: none !important;
            animation-name: none !important;
        }
        [class*="preloader"]:not(body):not(html),
        [id*="preloader"]:not(body):not(html),
        [class*="page-loader"]:not(body):not(html),
        [id*="page-loader"]:not(body):not(html),
        [class*="loading-screen"]:not(body):not(html),
        [id*="loading-screen"]:not(body):not(html),
        .rs-loader, .tp-loader, .sr7-loader,
        #cookiePopupHTMLAppContainer { display: none !important; }
        [id*="cookiePopup"]:not(body):not(html),
        [class*="cookiePopup"]:not(body):not(html),
        #cookiesEU-box, .ww_cookie_info { display: none !important; }
        .section_loader, .mfp-bg.mfp-ready { display: none !important; }
        body.themify-popup-showing { overflow: auto !important; }
        body.full-section-scrolling, body.full-section-scrolling-single,
        html.fp-scroll { overflow: auto !important; overflow-y: auto !important; }
        .fp-section-container { height: auto !important; min-height: 100vh !important; overflow: visible !important; position: relative !important; }
        html.has-scroll-init, html.has-scroll-smooth,
        [data-scroll-container], [data-scroll-section] {
            overflow: auto !important; overflow-y: auto !important;
            height: auto !important; position: relative !important;
            transform: none !important;
        }
        [data-scroll-section] {
            opacity: 1 !important; pointer-events: all !important;
            visibility: visible !important;
        }
        [data-scroll-speed], [data-scroll] { transform: none !important; }
        .c-scrollbar { display: none !important; }
        .pix-page-loading-bg, .pix-page-loading-bg::after { display: none !important; }
        .pix-footer-overlay { display: none !important; }
        body:not(.render) .pix-overlay-item,
        body:not(.pix-loaded) .pix-wml-header-btn {
            opacity: 1 !important; visibility: visible !important;
        }
        .pix-waiting, [data-anim-type], .animating-init, .animating,
        .pix-scale-in-sm, .slide-in-up, .slide-in-down,
        .slide-in-left, .slide-in-right, .fade-in, .fade-in-up,
        .fade-in-down, .zoom-in, .scale-in {
            visibility: visible !important; opacity: 1 !important;
            transform: none !important; animation-name: none !important;
        }
    `;
    const installStyle = () => {
        if (document.getElementById('__snap_init_style__')) return;
        const target = document.head || document.documentElement;
        if (!target) return false;
        const s = document.createElement('style');
        s.id = '__snap_init_style__';
        s.textContent = css;
        target.appendChild(s);
        return true;
    };
    if (!installStyle()) {
        document.addEventListener('DOMContentLoaded', installStyle);
    }

    // IntersectionObserver HIJACK
    try {
        const RealIO = window.IntersectionObserver;
        if (RealIO) {
            window.IntersectionObserver = class FakeIO {
                constructor(cb, opts) {
                    this.cb = cb;
                    this.opts = opts || {};
                    this._observed = new Set();
                }
                observe(el) {
                    if (!el || this._observed.has(el)) return;
                    this._observed.add(el);
                    const fire = () => {
                        try {
                            const rect = el.getBoundingClientRect ? el.getBoundingClientRect() : {};
                            this.cb([{
                                isIntersecting: true,
                                intersectionRatio: 1,
                                target: el,
                                boundingClientRect: rect,
                                intersectionRect: rect,
                                rootBounds: null,
                                time: performance.now(),
                            }], this);
                        } catch(e) {}
                    };
                    setTimeout(fire, 0);
                    setTimeout(fire, 200);
                }
                unobserve(el) { this._observed.delete(el); }
                disconnect() { this._observed.clear(); }
                takeRecords() { return []; }
            };
        }
    } catch(e) {}

    // Cookie consent flags
    try {
        const consent = {
            'cookieSettings': JSON.stringify({
                acceptedAll: true, acceptedTime: Date.now(),
                marketing: true, statistics: true, preferences: true, functional: true,
            }),
            'webwave-cookie-consent': 'accepted-all',
            'CookieConsent': '{stamp:%27snap%27%2Cnecessary:true%2Cpreferences:true%2Cstatistics:true%2Cmarketing:true%2Cver:1}',
            'OptanonAlertBoxClosed': new Date().toISOString(),
            'OptanonConsent': 'isGpcEnabled=0&datestamp=' + encodeURIComponent(new Date().toString()) + '&groups=C0001:1,C0002:1,C0003:1,C0004:1',
            'cookieconsent_status': 'allow',
            'cookies_accepted': 'true',
            'cookie_accepted': 'true',
            'gdpr_consent': 'true',
            'cc_cookie_accept': 'true',
            'cookiehub': '{"answered":true,"approved":["necessary","analytics","marketing","preferences"]}',
            'klaro': 'true',
            'borlabs-cookie': '{"consents":{"essential":["borlabs-cookie"]},"version":3}',
        };
        for (const [k, v] of Object.entries(consent)) {
            try { localStorage.setItem(k, v); } catch(e) {}
            try { sessionStorage.setItem(k, v); } catch(e) {}
        }
    } catch(e) {}

    // prefers-reduced-motion
    try {
        const origMM = window.matchMedia;
        window.matchMedia = function(q) {
            if (typeof q === 'string' && q.indexOf('prefers-reduced-motion') >= 0) {
                return {
                    matches: q.indexOf('reduce') >= 0,
                    media: q,
                    onchange: null,
                    addListener: () => {}, removeListener: () => {},
                    addEventListener: () => {}, removeEventListener: () => {},
                    dispatchEvent: () => false,
                };
            }
            return origMM.call(window, q);
        };
    } catch(e) {}
})();
"""

# ─── Helpers ──────────────────────────────────────────────────────────────────

def sanitize_name(name: str) -> str:
    return re.sub(r'[^\w\-.]', '_', name).strip('_')

def get_domain(url: str) -> str:
    return urlparse(url).netloc.replace('www.', '')

def get_subfolder_name(url: str) -> str:
    path = urlparse(url).path.strip('/')
    if not path:
        return 'homepage'
    name = sanitize_name(path.replace('/', '_'))
    return name[:80]

def get_zip_name(domain: str, mode: str = 'full') -> str:
    ts = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    d = sanitize_name(domain)
    if mode == 'screenshots':
        return f"SCREENSHOTS_{ts}_{d}.zip"
    if mode == 'crawl':
        return f"CRAWL_{ts}_{d}.zip"
    return f"FULL_{ts}_{d}.zip"

def get_screenshot_filename(url: str) -> str:
    domain = get_domain(url)
    sub = get_subfolder_name(url)
    date = datetime.now().strftime('%Y-%m-%d')
    name = f"{date}_{sanitize_name(domain)}"
    if sub != 'homepage':
        name += f"_{sub}"
    return f"{name}.png"

# ─── Sitemap/Crawl ────────────────────────────────────────────────────────────

SKIP_EXTENSIONS = {
    '.css', '.js', '.png', '.jpg', '.jpeg', '.gif', '.svg', '.webp', '.avif', '.ico',
    '.woff', '.woff2', '.ttf', '.eot', '.otf', '.mp4', '.webm', '.ogg', '.mp3',
    '.json', '.xml', '.txt', '.pdf', '.doc', '.docx', '.xls', '.xlsx', '.zip',
    '.rar', '.gz', '.map', '.less', '.scss',
}

def fetch_sitemap_urls(base_url: str, session: requests.Session) -> list:
    parsed = urlparse(base_url)
    origin = f"{parsed.scheme}://{parsed.netloc}"
    candidates = [
        urljoin(base_url, '/sitemap.xml'),
        urljoin(base_url, '/sitemap_index.xml'),
        urljoin(base_url, '/sitemap-index.xml'),
        urljoin(base_url, '/wp-sitemap.xml'),
    ]
    found = []
    for sm_url in candidates:
        try:
            r = session.get(sm_url, timeout=10, headers={'User-Agent': NORMAL_USER_AGENT})
            if r.status_code != 200:
                continue
            found.extend(_parse_sitemap_xml(r.text, origin))
        except Exception:
            continue
    return found

def _parse_sitemap_xml(xml_text: str, origin: str) -> list:
    urls = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return urls

    ns = ''
    if root.tag.startswith('{'):
        ns = root.tag.split('}')[0] + '}'

    for sm_element in root.findall(f'{ns}sitemap') + root.findall('sitemap'):
        loc = sm_element.find(f'{ns}loc') or sm_element.find('loc')
        if loc is not None and loc.text:
            sm_url = loc.text.strip()
            try:
                r = requests.get(sm_url, timeout=10, headers={'User-Agent': NORMAL_USER_AGENT})
                if r.status_code == 200:
                    urls.extend(_parse_sitemap_xml(r.text, origin))
            except Exception:
                pass

    for url_element in root.findall(f'{ns}url') + root.findall('url'):
        loc = url_element.find(f'{ns}loc') or url_element.find('loc')
        if loc is not None and loc.text:
            url = loc.text.strip()
            if url.startswith(origin):
                urls.append(url)

    return urls

def crawl_internal_links(page_url: str, session: requests.Session, max_pages: int = 50) -> list:
    parsed = urlparse(page_url)
    domain = parsed.netloc

    try:
        r = session.get(page_url, timeout=15, headers={'User-Agent': NORMAL_USER_AGENT})
        if r.status_code != 200:
            return [page_url]
    except Exception:
        return [page_url]

    found_urls = {page_url}
    href_pat = re.compile(r'href=["\']([^"\']+)["\']', re.IGNORECASE)
    seen = set()

    for m in href_pat.finditer(r.text):
        href = m.group(1).strip()
        if href.startswith(('#', 'mailto:', 'tel:', 'javascript:', 'data:')):
            continue
        full = urljoin(page_url, href)
        full, _ = urldefrag(full)
        fp = urlparse(full)
        if fp.scheme not in ('http', 'https'):
            continue
        if fp.netloc != domain:
            continue
        ext = Path(fp.path).suffix.lower()
        if ext in SKIP_EXTENSIONS:
            continue
        clean = full.rstrip('/')
        if clean not in seen:
            seen.add(clean)
            if len(seen) > max_pages:
                break
            found_urls.add(clean)

    return sorted(found_urls)

# ─── Popup killing ────────────────────────────────────────────────────────────

def close_popups(page):
    dismiss_selectors = [
        '#cookies-close-deny', '#cookies-close-accept', '#cookies-close-settings',
        '#CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll',
        '#CybotCookiebotDialogBodyLevelButtonLevelOptinDeclineAll',
        '.cc-dismiss', '.cc-btn.cc-dismiss',
        '.cookie-accept', '.cookie-close', '.consent-accept',
        '[class*="cookie"] .btn-primary', '[class*="cookie"] .btn-success',
        '[class*="consent"] .btn-primary', '[class*="gdpr"] .btn-primary',
        '[id*="cookie"] .btn-primary', '[id*="consent"] .btn-primary',
        '[id*="cookiePopup"] button:first-of-type',
        '.cookiePopup__button--accept', '.cookiePopup__acceptButton',
        'button.close', 'button[aria-label="Close"]', 'button[aria-label="close"]',
        'a.close', 'a[aria-label="Close"]', 'a[aria-label="close"]',
        '[data-dismiss="modal"]', '[data-bs-dismiss="modal"]',
        '.fancybox-close', '.fancybox-button--close', '.mfp-close',
        '[class*="close-btn"]', '[class*="closeBtn"]',
        '[class*="close-button"]', '[class*="popup-close"]',
        '[class*="modal-close"]',
        '[class*="newsletter"] [class*="close"]',
        '[class*="newsletter"] [class*="Close"]',
        '[id*="newsletter"] [class*="close"]',
        '[class*="popup"] [class*="close"]',
        '[class*="popup"] [class*="Close"]',
        '[id*="popup"] [class*="close"]',
        '[class*="overlay"] [class*="close"]',
        'svg[class*="close"]', 'svg[aria-label="Close"]', 'svg[aria-label="close"]',
        'img[alt="Close"]', 'img[alt="Zamknij"]', 'img[alt="zamknij"]',
        '.mfn-gdpr-button',
    ]
    for selector in dismiss_selectors:
        try:
            els = page.query_selector_all(selector)
            for el in els:
                if el.is_visible():
                    el.click()
                    page.wait_for_timeout(300)
        except Exception:
            pass

    accept_texts = [
        'Allow All', 'Accept All', 'Accept all', 'Akceptuj wszystkie',
        'Zezwól na wszystkie', 'Zaakceptuj wszystkie', 'Zaakceptuj',
        'Zgadzam się', 'Rozumiem', 'OK', 'Got it', 'I agree',
        'Allow all cookies', 'Akceptuj', 'Zezwól',
    ]
    for text in accept_texts:
        try:
            btn = page.get_by_role('button', name=text, exact=False).first
            if btn and btn.is_visible(timeout=200):
                btn.click(timeout=1000)
                page.wait_for_timeout(300)
                break
        except Exception:
            pass

    for _ in range(3):
        try:
            page.keyboard.press('Escape')
            page.wait_for_timeout(150)
        except Exception:
            pass

def _prepare_page(page, aggressive: bool = False):
    inject_anti_popup_css(page)
    _force_lazy_load(page)
    _finish_all_animations(page)
    _defuse_fullpage_sites(page)
    _defuse_locomotive_scroll(page)
    _defuse_pixfort(page)
    _walk_through_sliders(page)
    _wait_for_images(page)
    _wait_for_fonts(page)
    _scroll_and_wait(page)
    
    if aggressive:
        close_popups_aggressive(page)
    else:
        close_popups(page)

print("✅ Skrypt snap.py załadowany - gotowy do uruchomienia!")